# S2Gaussian Sequential Multi-Scene Kaggle Pipeline

This notebook trains all 8 scenes sequentially, renders predictions, and creates a submission CSV.

It includes a **resume mechanism** via `/kaggle/working/sequential_state.json` so rerunning after interruptions continues from the last completed stage.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import csv
import cv2
import torch

# ===== User-configurable paths =====
ROOT = Path('/kaggle/working')
INPUT_REPO = Path('/kaggle/input/datasets/kirankumarp05/kkk-ee-mcv/S2Gaussian')
DATA_ROOT = Path('/kaggle/input/datasets/kirankumarp05/ee5178-project/data')
DATA_PROCESSED_CANDIDATES = [
    Path('/kaggle/input/datasets/kirankumarp05/ee5178-project/data_processed'),
    Path('/kaggle/input/datasets/kirankumarp05/ee5178-project/data_processed_llff'),
    DATA_ROOT.parent / 'data_processed',
]

REPO = ROOT / 'S2Gaussian'
RUNS_ROOT = ROOT / 'runs_s2_seq'
LOG_ROOT = ROOT / 'logs_seq'
WORK_DATA_ROOT = ROOT / 'data_train_seq'
SUBMISSION_DIR = ROOT / 'submission_seq'
SUBMISSION_CSV = ROOT / 'submission_seq.csv'
STATE_FILE = ROOT / 'sequential_state.json'

REALESRGAN_DIR = ROOT / 'Real-ESRGAN'
REALESRGAN_WEIGHTS = REALESRGAN_DIR / 'RealESRGAN_x4plus.pth'

SCENES = ['aeroplane', 'bike', 'buddha', 'cycle', 'face', 'firehydrant', 'still3', 'toy']

for p in [RUNS_ROOT, LOG_ROOT, WORK_DATA_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('Configured scenes:', SCENES)
print('Runs root:', RUNS_ROOT)
print('State file:', STATE_FILE)

In [ ]:
def run(cmd: str):
    print(f'$ {cmd}')
    subprocess.run(cmd, shell=True, check=True)

def ensure_repo_and_deps():
    if INPUT_REPO.exists() and not REPO.exists():
        run(f'cp -r {INPUT_REPO} {REPO}')
    elif not REPO.exists():
        raise RuntimeError(f'S2Gaussian not found at {INPUT_REPO}')

    repo_code = REPO / 'fsgs' if (REPO / 'fsgs').exists() else REPO
    print('Using repo code:', repo_code)

    run('apt-get update -qq')
    run('apt-get install -y -qq build-essential ninja-build')

    req = repo_code / 'requirements.txt'
    if req.exists():
        run(f'pip install -r {req}')
    else:
        run('pip install torch torchvision basicsr realesrgan open3d timm plyfile opencv-python tqdm')

    dgr_path = repo_code / 'submodules' / 'diff-gaussian-rasterization-confidence'
    if not dgr_path.exists():
        dgr_path = repo_code / 'submodules' / 'diff-gaussian-rasterization'
    if dgr_path.exists():
        run(f'pip install --no-build-isolation {dgr_path}')

    knn_path = repo_code / 'submodules' / 'simple-knn'
    if not knn_path.exists():
        raise RuntimeError(f'simple-knn path missing: {knn_path}')

    os.environ['CUDA_HOME'] = '/usr/local/cuda'
    os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'
    os.environ['CUDACXX'] = '/usr/local/cuda/bin/nvcc'

    cu_file = knn_path / 'simple_knn.cu'
    txt = cu_file.read_text()
    if '#include <cfloat>' not in txt and '#include <float.h>' not in txt:
        lines = txt.splitlines()
        at = 0
        for i, line in enumerate(lines):
            if line.strip().startswith('#include'):
                at = i + 1
        lines.insert(at, '#include <cfloat>')
        cu_file.write_text('\n'.join(lines) + '\n')

    run(f'cd {knn_path} && rm -rf build dist *.egg-info')
    run(f'pip install --no-build-isolation -v {knn_path}')

    if not REALESRGAN_WEIGHTS.exists():
        found = None
        for root in [Path('/kaggle/input'), Path('/kaggle/working')]:
            if not root.exists():
                continue
            for cand in root.rglob('RealESRGAN_x4plus.pth'):
                found = cand
                break
            if found is not None:
                break
        if found is None:
            raise FileNotFoundError('RealESRGAN_x4plus.pth not found.')
        REALESRGAN_DIR.mkdir(parents=True, exist_ok=True)
        run(f'cp {found} {REALESRGAN_WEIGHTS}')

    import sys
    sys.path.insert(0, str(repo_code))
    from simple_knn._C import distCUDA2
    import diff_gaussian_rasterization

    return repo_code

REPO_CODE = ensure_repo_and_deps()
print('Environment setup complete.')

In [ ]:
import sys
sys.path.insert(0, str(REPO_CODE))
from scene.colmap_loader import read_extrinsics_binary, read_extrinsics_text

with open('test_filenames.json', 'r') as f:
    TEST_FILENAMES = json.load(f)
with open('target_sizes.json', 'r') as f:
    TARGET_SIZES = json.load(f)

def load_state():
    if STATE_FILE.exists():
        return json.loads(STATE_FILE.read_text())
    return {
        'scenes': {s: {'trained': False, 'rendered': False} for s in SCENES},
        'submission_built': False,
        'csv_built': False
    }

def save_state(state):
    STATE_FILE.write_text(json.dumps(state, indent=2))

def ensure_sparse0(scene_root: Path):
    sparse = scene_root / 'sparse'
    sparse0 = sparse / '0'
    if sparse0.exists():
        return
    req = ['cameras.bin', 'images.bin', 'points3D.bin']
    if sparse.exists() and all((sparse / n).exists() for n in req):
        sparse0.mkdir(parents=True, exist_ok=True)
        for n in req:
            shutil.copy2(sparse / n, sparse0 / n)

def read_sparse_image_names(scene_root: Path):
    images_bin = scene_root / 'sparse' / '0' / 'images.bin'
    images_txt = scene_root / 'sparse' / '0' / 'images.txt'
    if images_bin.exists():
        extr = read_extrinsics_binary(str(images_bin))
    elif images_txt.exists():
        extr = read_extrinsics_text(str(images_txt))
    else:
        raise FileNotFoundError(f'No images.bin/images.txt under {scene_root / "sparse/0"}')
    return sorted([Path(v.name).name for v in extr.values()])

def choose_scene_source(scene: str):
    candidates = [DATA_ROOT / scene]
    for pr in DATA_PROCESSED_CANDIDATES:
        candidates.append(pr / scene)

    valid = []
    for c in candidates:
        if not c.exists():
            continue
        ensure_sparse0(c)
        if (c / 'images').exists() and (c / 'sparse' / '0').exists():
            valid.append(c)

    if not valid:
        raise RuntimeError(f'No valid source candidate found for scene={scene}')

    sparse_names = read_sparse_image_names(valid[0])
    needed = set(sparse_names)

    for c in valid:
        names = set([p.name for p in (c / 'images').glob('*') if p.is_file()])
        if needed.issubset(names):
            return c

    return valid[0]

def prepare_writable_scene(scene: str):
    src = choose_scene_source(scene)
    dst = WORK_DATA_ROOT / scene
    if not dst.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
    ensure_sparse0(dst)
    probe = dst / 'sparse' / '0' / '.write_probe'
    probe.write_text('ok')
    probe.unlink()
    return src, dst

def train_scene(scene: str):
    src, dst = prepare_writable_scene(scene)
    model_path = RUNS_ROOT / scene
    model_path.mkdir(parents=True, exist_ok=True)
    log_file = LOG_ROOT / f'{scene}_train.log'

    num_gpus = torch.cuda.device_count()
    sr_gpu_id = 1 if num_gpus >= 2 else 0

    cmd = [
        'python', '-u', str(REPO / 'scripts' / 'train_s2gs.py'),
        '-s', str(dst),
        '-m', str(model_path),
        '--images', 'images',
        '--eval',
        '--sr_gpu_id', str(sr_gpu_id),
    ]

    print(f'[{scene}] source={src}')
    print(f'[{scene}] writable={dst}')
    print(f'[{scene}] model={model_path}')
    print(f'[{scene}] cmd={" ".join(cmd)}')

    with open(log_file, 'w') as log_f:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='')
            log_f.write(line)
        p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'Training failed for {scene}. See {log_file}')

def render_scene(scene: str):
    model_path = RUNS_ROOT / scene
    render_dir_10000 = model_path / 'test' / 'ours_10000' / 'renders'
    render_dir_30000 = model_path / 'test' / 'ours_30000' / 'renders'
    if (render_dir_10000.exists() and len(list(render_dir_10000.glob('*.png'))) > 0) or (render_dir_30000.exists() and len(list(render_dir_30000.glob('*.png'))) > 0):
        print(f'[{scene}] renders exist, skipping render.')
        return

    cmd = [
        'python', str(ROOT / 'gaussian-splatting' / 'render.py'),
        '--model_path', str(model_path),
        '--skip_train',
        '--quiet',
    ]
    print(f'[{scene}] render cmd={" ".join(cmd)}')
    subprocess.run(cmd, check=True)

def build_submission_folder():
    if SUBMISSION_DIR.exists():
        shutil.rmtree(SUBMISSION_DIR)
    SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

    for scene in SCENES:
        dst_scene = SUBMISSION_DIR / scene
        dst_scene.mkdir(parents=True, exist_ok=True)

        model_path = RUNS_ROOT / scene
        render_dir = model_path / 'test' / 'ours_10000' / 'renders'
        if not render_dir.exists():
            render_dir = model_path / 'test' / 'ours_30000' / 'renders'

        if not render_dir.exists():
            raise FileNotFoundError(f'Render dir missing for {scene}: {render_dir}')

        expected = TEST_FILENAMES[scene]
        for idx, target_name in enumerate(expected):
            src = render_dir / f'{idx:05d}.png'
            if not src.exists():
                raise FileNotFoundError(f'Missing render {src}')
            out = dst_scene / target_name
            shutil.copy2(src, out)

def enforce_target_sizes():
    for scene, (tw, th) in TARGET_SIZES.items():
        scene_dir = SUBMISSION_DIR / scene
        for p in scene_dir.iterdir():
            if p.suffix.lower() not in {'.png', '.jpg', '.jpeg'}:
                continue
            img = cv2.imread(str(p))
            if img is None:
                raise RuntimeError(f'Unreadable image: {p}')
            h, w = img.shape[:2]
            if (w, h) != (tw, th):
                img = cv2.resize(img, (tw, th), interpolation=cv2.INTER_LANCZOS4)
                cv2.imwrite(str(p), img)

def write_submission_csv():
    rows = []
    for scene in sorted(SCENES):
        scene_dir = SUBMISSION_DIR / scene
        files = sorted([p for p in scene_dir.iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
        for p in files:
            img = cv2.imread(str(p))
            if img is None:
                raise RuntimeError(f'Unreadable image: {p}')
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            rows.append([
                f'{scene}_{p.stem}',
                scene,
                p.stem,
                ' '.join(map(str, img.flatten())),
            ])

    rows.sort(key=lambda r: (r[1], r[2]))
    with open(SUBMISSION_CSV, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['id', 'scene', 'image_name', 'image'])
        w.writerows(rows)

    print('CSV written:', SUBMISSION_CSV, 'rows=', len(rows))

print('Helpers loaded.')

In [ ]:
state = load_state()
print('Loaded state:')
print(json.dumps(state, indent=2))

# ===== Stage 1: train all scenes sequentially (resume-safe) =====
for scene in SCENES:
    if state['scenes'][scene]['trained']:
        print(f'[SKIP][TRAIN] {scene}')
        continue
    print(f'\n[RUN][TRAIN] {scene}')
    train_scene(scene)
    state['scenes'][scene]['trained'] = True
    save_state(state)

print('\nTraining stage complete.')

In [ ]:
state = load_state()

# ===== Stage 2: render all scenes (resume-safe) =====
for scene in SCENES:
    if state['scenes'][scene]['rendered']:
        print(f'[SKIP][RENDER] {scene}')
        continue
    print(f'\n[RUN][RENDER] {scene}')
    render_scene(scene)
    state['scenes'][scene]['rendered'] = True
    save_state(state)

print('\nRender stage complete.')

In [ ]:
state = load_state()

# ===== Stage 3: package submission (resume-safe) =====
if not state['submission_built']:
    build_submission_folder()
    enforce_target_sizes()
    state['submission_built'] = True
    save_state(state)
    print('Submission folder built:', SUBMISSION_DIR)
else:
    print('[SKIP] submission folder already built')

if not state['csv_built']:
    write_submission_csv()
    state['csv_built'] = True
    save_state(state)
else:
    print('[SKIP] csv already built')

print('\nFinal state:')
print(json.dumps(state, indent=2))

In [ ]:
# Final quick validation
with open('test_filenames.json', 'r') as f:
    expected = json.load(f)

all_ok = True
for scene in SCENES:
    cnt = len([p for p in (SUBMISSION_DIR / scene).iterdir() if p.suffix.lower() in {'.png', '.jpg', '.jpeg'}])
    exp = len(expected[scene])
    print(f'{scene}: expected={exp} actual={cnt}')
    if cnt != exp:
        all_ok = False

print('submission_dir:', SUBMISSION_DIR)
print('submission_csv:', SUBMISSION_CSV)
print('ALL_OK=', all_ok)